# k3-e5 — Task C1: Mã hoá mE5-large cho đoạn cha + đoạn con (GPU T4 ×2 độc lập của Role C)

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On.
**Add Input:** `mir2-code` (bản mới nhánh c-e5), `mir2-data`, `mir2-chunks` (output k1 của team).

**Tài nguyên & Phân việc Role C:**
- Role C chạy độc lập trên 1 phiên Kaggle 2 GPU T4 (`MEMBERS = 1`, `MEMBER = 0`).
- GPU 0 chạy shard `0/2`, GPU 1 chạy shard `1/2` $\rightarrow$ Chạy song song hoàn thành 100% kho của C.
- **Mốc Go/No-Go (Hạn chót 21/10):** Ô (2) chạy lệnh `profile` đo thời gian mẫu thực tế và đưa ra quyết định tự động.
- **Fallback:** Nếu NO-GO (nguy cơ quá 30h quota hoặc trễ hạn 24/10), đổi sang `LEVELS = ['parents']` (Option A).
**Đầu ra:** `emb_e5/{parents,children}/<khối>/` $\rightarrow$ Tạo dataset **`mir2-emb-e5`**.

In [ ]:
# (1) Chuẩn bị môi trường & cấu hình mE5-large từ GitHub nhánh c-e5
NB_VERSION = 'k3-e5-v1 (20/10/2026)'
MEMBER = 0           # Role C chạy độc lập: 1 thành viên xử lý toàn bộ kho
MEMBERS = 1          # 2 GPU chạy shard 0/2 và 1/2
LEVELS = ['parents', 'children']  # Nếu NO-GO -> đổi thành ['parents']
SKIP_FROM = []
RUN_PROFILE = True   # Bật đo tốc độ & đánh giá Go/No-Go
QUOTA_GPU_HOURS = 30.0

import os, sys, shutil, subprocess

KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')
CODE = f'{KAGGLE}/working/code'

# Chuyển ra ngoài /kaggle trước khi xoá để tránh lỗi Linux getcwd()
os.chdir(KAGGLE)
shutil.rmtree(CODE, ignore_errors=True)

# Kéo trực tiếp code Task C1 mới nhất từ GitHub nhánh c-e5 (không cần file mir2-code)
print("-> Đang tải mã nguồn nhánh c-e5 từ GitHub...")
res = subprocess.run(['git', 'clone', '--depth', '1', '-b', 'c-e5',
                      'https://github.com/nhantranduynhan11-del/Cross-Lingual-Medical-Retrieval.git', CODE],
                     capture_output=True, text=True)
if res.returncode != 0:
    print("Git clone log:", res.stderr)
    raise RuntimeError(f"Không thể tải code từ GitHub: {res.stderr}")
print("-> Đã tải xong mã nguồn nhánh c-e5!")

os.chdir(CODE)
os.environ['PYTHONPATH'] = f"{CODE}:{os.environ.get('PYTHONPATH', '')}"
if CODE not in sys.path:
    sys.path.insert(0, CODE)

from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {
    'encode.model': 'intfloat/multilingual-e5-large',
    'encode.out_dir': 'emb_e5'
}, inputs=('chunks',))

In [ ]:
# (2) Speed Profiling & Đánh giá mốc Go/No-Go
import os, sys
os.chdir('/kaggle/working/code')
os.environ['PYTHONPATH'] = f"/kaggle/working/code:{os.environ.get('PYTHONPATH', '')}"

prev = [str(d / 'emb_e5') for d in locate_all('emb_e5')]
print('emb_e5 trong input:', prev)
old = [p for p in prev if any(f'/{x}/' in p + '/' for x in SKIP_FROM)]
mine_prev = [p for p in prev if p not in old]
skip = f' --skip {",".join(old)}' if old else ''
resume = f' --resume {",".join(mine_prev)}' if mine_prev else ''
print('bỏ qua (bản cũ):', old, '| chạy tiếp từ:', mine_prev)

if RUN_PROFILE:
    print('=== [TASK C1] CHẠY SPEED PROFILING & ĐÁNH GIÁ GO / NO-GO ===')
    sh(f'python -m mirret.encode profile --config {K.cfg} --device cuda:0 --n-children 2000 --n-parents 1000 --quota {QUOTA_GPU_HOURS}')

for lv in LEVELS:
    sh(f'python -m mirret.encode plan --config {K.cfg} --level {lv}{skip}')

In [ ]:
# (3) Chạy mã hoá 100% Corpus trên 2 GPU T4 song song
# GPU 0: Shard 0/2; GPU 1: Shard 1/2
n = 2 * MEMBERS
for lv in LEVELS:
    par([f'CUDA_VISIBLE_DEVICES={g} python -m mirret.encode run --config {K.cfg} --level {lv} '
         f'--shard {g}/{n} --device cuda:0{skip}{resume}' for g in range(2)], f'encode_{lv}')
report('k3_e5')